# LLM Self-Hosting: The Full Deployment Landscape

A theory-only reference notebook mapping the complete landscape for self-hosting
open-source LLMs (Llama, Qwen, Mistral, and similar) — from a laptop demo all the way to
a multi-tenant Kubernetes cluster.

### Table of Contents
1. [The Big Picture](#1)
2. [Local Hosting](#2)
3. [Quantization — Making Models Fit](#3)
4. [Online Deployment: Serving Engines](#4)
5. [Online Deployment: Serving Frameworks / API Layer](#5)
6. [Managed Deployment Services](#6)
7. [High-Scale Deployment: Kubernetes + llm-d](#7)
8. [The Supporting Layers Around Serving](#8)
9. [Full Landscape Map](#9)
10. [Glossary — Every Acronym, Spelled Out](#10)


## 1. The Big Picture <a id="1"></a>

Self-hosting an open-source **LLM (Large Language Model)** — Llama, Qwen, Mistral, and
similar — means running the model's weights on infrastructure you control, instead of
calling a provider's API. The landscape splits along two axes:

- **Where** the model runs: a single local machine vs. a fleet of networked GPUs.
- **What surrounds the raw model**: the inference engine, the API layer on top of it,
  who manages the hardware, and (at scale) who orchestrates the fleet.

```
   Local Hosting  -------------->  Online Deployment
   (demo / hobby /                 (real traffic, multiple users,
    quick testing)                  uptime & throughput matter)
                                             |
                          +------------------+------------------+
                          v                  v                  v
                  Serving Engine     Serving Framework    Who runs the
                  (does the           (exposes an API      hardware
                   computation)        around the engine)  (self-managed ->
                                                             managed service ->
                                                             orchestrated cluster)
```


## 2. Local Hosting <a id="2"></a>

For demos, hobby projects, or quick local testing — no networked cluster, no managed
infra, just one machine.

| Tool | What it is | Strengths |
|---|---|---|
| **Ollama** | A wrapper around **llama.cpp** with a simple CLI/API and automatic model management | Easiest on-ramp — one command pulls and runs a model, sensible defaults, built-in model registry |
| **llama.cpp** | A C/C++ inference runtime originally built for running LLMs efficiently on consumer hardware | Runs **quantized (GGUF)** models on **CPU or GPU** (CUDA/Metal/Vulkan offload) — not CPU-only; its real strength is squeezing a model onto hardware that couldn't run the full-precision version at all |

Both tools depend on **quantization** to make a model small enough to run locally at all
— see [Section 3](#3).


## 3. Quantization -- Making Models Fit <a id="3"></a>

A model's raw weights are typically stored in **FP16 (16-bit Floating Point)** or
**BF16 (Brain Floating Point, 16-bit)**. Quantization compresses those weights to fewer
bits, trading a small amount of accuracy for a large drop in memory footprint — often the
difference between "doesn't fit on this GPU" and "fits comfortably."

| Format | Full form | Typically used with | Notes |
|---|---|---|---|
| **GGUF** | GPT-Generated Unified Format | llama.cpp, Ollama | The file format itself; bundles quantized weights + metadata in one file; supports many bit-widths (2-bit to 8-bit) |
| **AWQ** | Activation-aware Weight Quantization | vLLM, TensorRT-LLM | Protects the weights that matter most for activations, keeping accuracy closer to the original model |
| **GPTQ** | Post-training quantization method for GPT-style models (from the paper *"GPTQ: Accurate Post-Training Quantization for Generative Pre-trained Transformers"*) | vLLM, TensorRT-LLM | One of the earliest popular 4-bit quantization methods for transformers |
| **FP8** | 8-bit Floating Point | vLLM, SGLang, TensorRT-LLM | Native hardware support on recent NVIDIA GPUs (Hopper/Blackwell); good accuracy-to-speed tradeoff for serving, not just storage |

**Rule of thumb:** GGUF is the local/CPU-friendly lane (llama.cpp/Ollama); AWQ/GPTQ/FP8 are
the GPU-serving lane (vLLM/SGLang/TensorRT-LLM).


## 4. Online Deployment: Serving Engines <a id="4"></a>

A **serving engine** is the component that actually does the computation — it owns the
attention algorithm, the **KV (Key-Value) cache** management, and the batching strategy
that determines how many concurrent requests a GPU can actually handle.

| Engine | Full form / origin | Core technique | What it's known for |
|---|---|---|---|
| **vLLM** | Informally "Virtual Large Language Model" — the "v" nods to *virtual memory* (no single official expansion) | **PagedAttention** — manages the KV cache in non-contiguous memory "pages," the same idea as OS virtual memory paging, to eliminate memory waste | High throughput, continuous batching, the de-facto default open-source engine |
| **SGLang** | Structured Generation Language | **RadixAttention** — stores the KV cache in a radix tree so requests sharing a common prefix (e.g. the same system prompt) automatically reuse cached computation | Strong for structured/constrained output (JSON schemas, regex-constrained decoding) and complex multi-call/agentic prompting patterns |
| **TensorRT-LLM** | TensorRT for Large Language Models (TensorRT = NVIDIA's Tensor Runtime, a deep-learning inference optimizer/compiler) | Compiles the model into a hardware-specific optimized engine for NVIDIA GPUs | Typically the ceiling for raw throughput/latency on NVIDIA hardware, at the cost of being NVIDIA-only and needing a compile step per model/GPU combo |

**Continuous batching** (used by vLLM and SGLang) means new requests join a running batch
as soon as a GPU slot frees up, instead of waiting for the whole batch to finish — this is
what makes GPU utilization high under real, staggered traffic.


## 5. Online Deployment: Serving Frameworks / API Layer <a id="5"></a>

A **serving framework/layer** is what exposes the engine to the outside world — an HTTP
API, request routing, and process management. The key nuance: **engine and API layer are
sometimes the same package, sometimes separate.**

| Tool | Relationship to the engine | Notes |
|---|---|---|
| **vLLM** | Bundles engine + an OpenAI-compatible API server in one package | `vllm serve <model>` starts both at once |
| **SGLang** | Same bundled pattern — engine + OpenAI-compatible server together | `sglang.launch_server` |
| **TensorRT-LLM** | Now ships its **own** native OpenAI-compatible server (`trtllm-serve`) | Previously depended on Triton for serving; as of recent versions it no longer needs Triton for a basic deployment |
| **Triton Inference Server** | A general-purpose, multi-framework model server (NVIDIA) | Can serve TensorRT-LLM, vLLM, ONNX, and custom Python backends — one option among several for TensorRT-LLM, not a hard dependency anymore |

**Why this distinction matters:** vLLM and SGLang are easy to reach for precisely
*because* they collapse "engine" and "API layer" into one install — TensorRT-LLM used to
require pairing with Triton to get an API at all, but that gap has closed.


## 6. Managed Deployment Services <a id="6"></a>

Once you don't want to manage GPU hardware yourself, there's a spectrum of "how much
infrastructure do I hand off" — not one flat category:

| Tier | Examples | What you get | What you still manage |
|---|---|---|---|
| **Serverless GPU platforms** | RunPod, Modal, Baseten, Replicate, Beam | Pay-per-use GPU compute, spin up/down automatically, often has templates for popular engines | Your own serving container/config (though many offer one-click vLLM deploys) |
| **Managed enterprise AI hubs** | HF (Hugging Face) Inference Endpoints, SageMaker (AWS), Vertex AI (Google Cloud), Azure ML | Fuller lifecycle management (versioning, autoscaling, monitoring) integrated with the cloud provider's ecosystem | Less low-level control, generally higher cost floor |
| **Raw IaaS (Infrastructure as a Service)** | Lambda Labs, CoreWeave, FluidStack, traditional cloud VMs | Just the GPU box — you install and run vLLM/SGLang/TensorRT-LLM yourself | Everything above the bare metal/VM |

**RunPod and Modal** (your original examples) sit specifically in the **serverless**
tier — cheapest way to get GPU time without owning hardware, but you're still the one
choosing and configuring the serving engine on top.


## 7. High-Scale Deployment: Kubernetes + llm-d <a id="7"></a>

At real production scale — many models, many tenants, GPUs as the scarce/expensive
resource — a plain Kubernetes deployment of vLLM replicas hits a ceiling: **Kubernetes
doesn't understand LLM-specific state** (KV cache locality, prefill vs. decode compute
profiles, multi-GPU model sharding).

**llm-d** is the orchestration layer that adds this awareness on top of Kubernetes:

| Component | What it does |
|---|---|
| **IGW (Inference Gateway)** | Routes requests to the pod that already holds the relevant KV cache, instead of round-robin |
| **InferencePool** (a **CRD**, Custom Resource Definition) | Declares a pool of pods serving one model + engine combination |
| **P/D (Prefill/Decode) disaggregation** | Runs the compute-heavy prefill phase and the memory-bandwidth-heavy decode phase on separately-provisioned pods |
| **LWS (LeaderWorkerSet)** | Coordinates multi-GPU, multi-node serving for models too large for one machine |

llm-d is a **CNCF (Cloud Native Computing Foundation) Sandbox project**, built on top of
**vLLM** as the reference engine and the **Kubernetes Gateway API Inference Extension** as
the routing spec — it's the layer that turns a fleet of independent vLLM replicas into one
cache-aware, phase-aware cluster.


## 8. The Supporting Layers Around Serving <a id="8"></a>

Serving the model is necessary but not sufficient for production. A few adjacent layers
sit around every option above, regardless of which tier you're in:

- **Gateway/routing** — LiteLLM, Portkey, OpenRouter: route requests across multiple
  models/providers behind one API, handle fallback and load balancing.
- **Observability** — Langfuse, LangSmith, Arize Phoenix: trace and monitor requests once
  they're live.
- **Caching** — semantic caching and prefix/KV caching (the same idea PagedAttention and
  RadixAttention apply, extended cluster-wide by llm-d's IGW).
- **Guardrails/security** — Llama Guard, NeMo Guardrails, LLM Guard: filter inputs/outputs
  in production.

(These are covered in depth in your existing "9 missing LLMOps pieces" notebook —
[[ai-engineering-knowledge-base]].)


## 9. Full Landscape Map <a id="9"></a>

```
                              SELF-HOSTING AN OPEN-SOURCE LLM
                                    (Llama / Qwen / Mistral)
                                             |
                 +---------------------------+---------------------------+
                 v                                                       v
          LOCAL HOSTING                                          ONLINE DEPLOYMENT
   (demo / hobby / quick test)                              (real traffic, multiple users)
                 |                                                       |
        +--------+--------+                          +------------------+------------------+
        v                 v                          v                  v                  v
     Ollama          llama.cpp                 SERVING ENGINE    SERVING FRAMEWORK      WHO RUNS
   (easiest,      (CPU or GPU,                 (does compute)    (exposes the API)     THE HARDWARE
  wraps llama.cpp)  quantized GGUF)                  |                  |                   |
        |                 |                    vLLM (PagedAttn)  vLLM (bundled)    +--------+--------+
        +--------+--------+                    SGLang (RadixAttn) SGLang (bundled)  v        v        v
                 v                              TensorRT-LLM      trtllm-serve /  Serverless Managed  Raw
          needs QUANTIZATION                                       Triton        (RunPod,   Hub    IaaS
       (GGUF / AWQ / GPTQ / FP8)                                                   Modal) (SageMaker (Lambda
                                                                                             /Vertex) Labs,
                                                                                                      CoreWeave)
                                                                          |
                                                                          v
                                                           HIGH-SCALE ORCHESTRATION
                                                             Kubernetes + llm-d
                                                       (IGW cache-aware routing, P/D
                                                        disaggregation, LWS multi-node)
                                                                          |
                                                                          v
                                                         SUPPORTING LAYERS (all tiers)
                                                   gateways . observability . caching . guardrails
```


## 10. Glossary -- Every Acronym, Spelled Out <a id="10"></a>

| Acronym | Full form |
|---|---|
| **LLM** | Large Language Model |
| **GPU** | Graphics Processing Unit |
| **CPU** | Central Processing Unit |
| **API** | Application Programming Interface |
| **KV cache** | Key-Value cache |
| **FP16 / FP8** | 16-bit / 8-bit Floating Point |
| **BF16** | Brain Floating Point, 16-bit |
| **GGUF** | GPT-Generated Unified Format |
| **AWQ** | Activation-aware Weight Quantization |
| **GPTQ** | Not a strict acronym -- post-training quantization method introduced in *"GPTQ: Accurate Post-Training Quantization for Generative Pre-trained Transformers"* |
| **vLLM** | Informally "Virtual Large Language Model" (no single official expansion; "v" nods to virtual memory) |
| **SGLang** | Structured Generation Language |
| **TensorRT-LLM** | TensorRT for Large Language Models |
| **TensorRT** | NVIDIA's Tensor Runtime (deep-learning inference optimizer/compiler) |
| **IaaS** | Infrastructure as a Service |
| **HF** | Hugging Face |
| **CNCF** | Cloud Native Computing Foundation |
| **CRD** | Custom Resource Definition |
| **IGW** | Inference Gateway |
| **LWS** | LeaderWorkerSet |
| **P/D** | Prefill / Decode |
